# Hands-On 2: Decision trees and estimated performance

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, RocCurveDisplay
from mlcourse.labs import split_classification
from mlcourse.widgets import interactive_tree
from sklearn.metrics import roc_auc_score

## 1. Prepare a classification task

Load the reservations data. Select the numeric predictors and encode the target.

**Prediction:** Why exclude `booking_status` from the predictors?

*Your response.*

In [ ]:
data = load_course_data('hotel')
features = [c for c in data if c.startswith('no_of_')] + ['required_car_parking_space', 'lead_time', 'repeated_guest', 'avg_price_per_room']
X = data[features]
y = data.booking_status.eq('Canceled').astype(int)
X_train, X_test, y_train, y_test = split_classification(X, y)
display(X.head())
display(pd.DataFrame({'train_count': y_train.value_counts(), 'test_count': y_test.value_counts()}))

**Observation:** Which information is available to the classifier?

*Your response.*

**Explanation:** Explain the roles of predictors and target.

*Your response.*

## 2. Inspect a small tree

Fit `DecisionTreeClassifier(max_depth=3)`. Display its first two levels.

**Prediction:** What shape does a split on one feature create?

*Your response.*

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(15, 7), layout='constrained')
plot_tree(tree, feature_names=features, class_names=['Not canceled', 'Canceled'],
          max_depth=2, filled=True, rounded=True, fontsize=8, ax=ax)
plt.show()
print(f'Train accuracy: {accuracy_score(y_train, tree.predict(X_train)):.3f}')
print(f'Test accuracy: {accuracy_score(y_test, tree.predict(X_test)):.3f}')

**Observation:** Trace one sequence of split conditions.

*Your response.*

**Explanation:** How do the conditions define a region of feature space?

*Your response.*

## 3. Inspect classification errors

Display the confusion matrix and ROC curve for the same fitted tree.

**Prediction:** Can changing the probability threshold change predictions without fitting another tree?

*Your response.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')
ConfusionMatrixDisplay.from_estimator(tree, X_test, y_test, ax=axes[0], colorbar=False)
RocCurveDisplay.from_estimator(tree, X_test, y_test, ax=axes[1])
axes[1].get_legend().remove()
print(f'ROC AUC: {roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]):.3f}')
plt.show()

**Observation:** Which errors appear in the off-diagonal cells?

*Your response.*

**Explanation:** Relate the ROC curve to changing the threshold.

*Your response.*

## 4. Estimate variability

Compute five stratified cross-validation scores using the training partition.

**Prediction:** Will the five scores be identical?

*Your response.*

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
fold_scores = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=0), X_train, y_train, cv=cv)
display(pd.DataFrame({'fold': np.arange(1, 6), 'accuracy': fold_scores}))
print(f'Mean accuracy: {fold_scores.mean():.3f}; standard deviation: {fold_scores.std(ddof=1):.3f}')

**Observation:** Describe the spread of the scores.

*Your response.*

**Explanation:** Why can a fixed configuration have different estimated performance across folds?

*Your response.*

## 5. Inspect sample size and capacity

Compare shallow and deeper trees at three training-set sizes.

**Prediction:** Which configuration should fit small training samples most closely?

*Your response.*

In [ ]:
rows = []
for depth in [3, 10]:
    sizes, train_scores, validation_scores = learning_curve(
        DecisionTreeClassifier(max_depth=depth, random_state=0), X_train, y_train,
        train_sizes=[.2, .6, 1.0], cv=3, shuffle=True, random_state=0)
    for size, train_score, validation_score in zip(sizes, train_scores.mean(axis=1), validation_scores.mean(axis=1)):
        rows.append({'max_depth': depth, 'n_training': size, 'train_accuracy': train_score, 'validation_accuracy': validation_score})
curve = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')
for depth, group in curve.groupby('max_depth'):
    ax.plot(group.n_training, group.train_accuracy, '--o', label=f'Train, depth {depth}')
    ax.plot(group.n_training, group.validation_accuracy, '-s', label=f'Validation, depth {depth}')
ax.set(xlabel='Training observations', ylabel='Accuracy', title='Learning curves')
ax.legend()
plt.show()
display(curve)

**Observation:** Compare the train-validation gaps.

*Your response.*

**Explanation:** Explain the gaps using model capacity and sample size.

*Your response.*

## 6. Manipulate tree geometry

Use `ds2` to vary `max_depth` and `min_samples_leaf` in a two-dimensional feature space.

**Prediction:** How should increasing `min_samples_leaf` affect small regions?

*Your response.*

In [ ]:
geometry = load_course_data('ds2')
a, b, c, d = split_classification(geometry[['X1', 'X2']], geometry.Class)
tree_lab = interactive_tree(a, b, c, d)
display(tree_lab.widget)

**Observation:** Record a setting that removes small regions.

*Your response.*

**Explanation:** Explain why depth and leaf size both constrain capacity.

*Your response.*